In [44]:
# ==========================================
# Step 1: Import Libraries
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [45]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

In [46]:
# ==========================================
# Step 2: Load Dataset
# ==========================================
df = pd.read_csv("movies.csv")

print("Shape :", df.shape)
print("Columns :", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicate Rows :", df.duplicated().sum())

Shape : (500, 22)
Columns : ['Unnamed: 0', 'título', 'título_original', 'año', 'duración', 'país', 'dirección', 'guion', 'reparto', 'productora', 'género', 'sinopsis', 'premios', 'puntuación_fa', 'votos_fa', 'puntuación_positiva', 'puntuación_neutral', 'puntuación_negativa', 'portada_url', 'portada_local', 'puntuación_imdb', 'votos_imdb']

Missing Values:
 Unnamed: 0               0
título                   0
título_original          0
año                      0
duración                 2
país                     0
dirección                0
guion                    0
reparto                  0
productora               1
género                   0
sinopsis                 0
premios                117
puntuación_fa            0
votos_fa                 0
puntuación_positiva     76
puntuación_neutral      76
puntuación_negativa     76
portada_url              0
portada_local            0
puntuación_imdb         37
votos_imdb              37
dtype: int64

Duplicate Rows : 0


In [47]:
# ==========================================
# Step 3: Drop Irrelevant Columns
# ==========================================
df = df.drop([
    'Unnamed: 0',
    'título',
    'título_original',
    'sinopsis',
    'portada_url',
    'portada_local'
], axis=1)

In [48]:
# ==========================================
# Step 4: Handle Missing Values
# ==========================================
for col in df.select_dtypes(include=['int64','float64']).columns:
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include=['object']).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\admin\AppData\Local\Temp\ipykernel_13604\189781366.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\admin\AppData\Local\Temp\ipykernel_13604\189781366.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

In [49]:
# ==========================================
# Step 5: Encode Categorical Features
# ==========================================
label_encoders = {}

for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [50]:
# ==========================================
# Step 6: Feature Selection
# ==========================================
target_column = "puntuación_imdb"

X = df.drop(target_column, axis=1)
y = df[target_column]


In [51]:
# ==========================================
# Step 7: Feature Scaling
# ==========================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [52]:
# ==========================================
# Step 8: Train-Test Split
# ==========================================
x_train, x_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

In [53]:
# ==========================================
# Step 9: Evaluation Function
# ==========================================
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE :", rmse)
    print("R2 Score :", r2)

In [54]:
print("\n===== Linear Regression =====")

lr = LinearRegression()
lr.fit(x_train, y_train)

y_pred = lr.predict(x_test)

evaluate_model(y_test, y_pred)


===== Linear Regression =====
MAE : 0.14932905898910007
MSE : 0.04124353380578232
RMSE : 0.20308504082226816
R2 Score : 0.6412482707127249


In [55]:
print("\n===== Polynomial Regression =====")

poly = PolynomialFeatures(degree=2)

x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

poly_model = LinearRegression()
poly_model.fit(x_train_poly, y_train)

y_pred = poly_model.predict(x_test_poly)

evaluate_model(y_test, y_pred)


===== Polynomial Regression =====
MAE : 0.18444691672029417
MSE : 0.05775576758902403
RMSE : 0.2403242967097252
R2 Score : 0.4976186668085315


In [56]:
#Ride Regression
ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)

y_pred = ridge.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.14932682476074702
MSE : 0.04125990270975963
RMSE : 0.20312533743912803
R2 Score : 0.641105887845242


In [57]:
#Lasso Regression
lasso = Lasso(alpha=0.1)
lasso.fit(x_train, y_train)

y_pred = lasso.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.19408763967335815
MSE : 0.07470968139476675
RMSE : 0.2733307179860448
R2 Score : 0.35014716437522453


In [58]:
#Elastic Regression
elastic = ElasticNet(alpha=0.1)
elastic.fit(x_train, y_train)

y_pred = elastic.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.16702887377705544
MSE : 0.054338382252560144
RMSE : 0.23310594641184113
R2 Score : 0.5273443664750695


In [59]:
#Decision Regression
dt = DecisionTreeRegressor(random_state=42)
dt.fit(x_train, y_train)

y_pred = dt.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.206
MSE : 0.07900000000000004
RMSE : 0.281069386451104
R2 Score : 0.3128283636616681


In [60]:
#Random Regression
rf = RandomForestRegressor(random_state=42)
rf.fit(x_train, y_train)

y_pred = rf.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.13648999999999986
MSE : 0.034864649999999935
RMSE : 0.18672078084669616
R2 Score : 0.696734195052365


In [61]:
#SVR Regeression
svr = SVR()
svr.fit(x_train, y_train)

y_pred = svr.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.16197927196668688
MSE : 0.04586953973943317
RMSE : 0.21417175289807283
R2 Score : 0.6010095356856657


In [62]:
#KNN regression
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(x_train, y_train)

y_pred = knn.predict(x_test)

evaluate_model(y_test, y_pred)

MAE : 0.17180000000000015
MSE : 0.05758800000000013
RMSE : 0.2399749986977813
R2 Score : 0.4990779722347858
